In [ ]:
import json
import pandas as pd
from datetime import time

config = json.load(open("config.json"))
bars = pd.concat([pd.read_csv(f"data/raw/bars_{t}.csv") for t in config["tickers"]], ignore_index=True)
print(len(bars), "raw rows")

174578 raw rows


In [4]:
bars["bar_start_utc"] = pd.to_datetime(bars["timestamp"], utc=True)
bars["ny_time"] = bars["bar_start_utc"].dt.tz_convert("America/New_York")
bars = bars[(bars["ny_time"].dt.time >= time(9, 30)) & (bars["ny_time"].dt.time < time(16, 0))].copy()
bars["session_date"] = bars["ny_time"].dt.date
print(len(bars), "regular-hours rows")

115033 regular-hours rows


In [5]:
dups = bars.duplicated(subset=["symbol", "bar_start_utc"]).sum()
bad_price = (bars[["open", "high", "low", "close"]] <= 0).any(axis=1).sum()
bad_range = ((bars["low"] > bars[["open", "close"]].min(axis=1)) |
             (bars["high"] < bars[["open", "close"]].max(axis=1))).sum()
bad_volume = (bars["volume"] < 0).sum()
print("Duplicates:", dups, "| Bad prices:", bad_price, "| Bad ranges:", bad_range, "| Bad volume:", bad_volume)

Duplicates: 0 | Bad prices: 0 | Bad ranges: 0 | Bad volume: 0


In [6]:
minutes = bars["ny_time"].dt.hour * 60 + bars["ny_time"].dt.minute
bars["session_stage"] = pd.cut(minutes, bins=[570, 600, 690, 810, 930, 960],
                               labels=["Opening", "Morning", "Midday", "Afternoon", "Closing"], right=False)
bars["session_stage"].value_counts(sort=False)

session_stage
Opening       8850
Morning      26546
Midday       35391
Afternoon    35396
Closing       8850
Name: count, dtype: int64

In [7]:
bars["feed"] = "sip"
cols = ["symbol", "feed", "bar_start_utc", "session_date", "session_stage",
        "open", "high", "low", "close", "volume", "trade_count", "vwap"]
bars[cols].to_csv("data/processed/bars_clean.csv", index=False)
print("Saved", len(bars), "rows")

Saved 115033 rows
